# 00 · Quickstart (run top to bottom in the first 5 minutes)

**Who:** run it on the real data first.
**Gives:** format check → event study → baselines on the team folds → industry filter → a first upload file.

If any check in section 1 fails, the format changed. Shout, then adapt `SPEC` / the loader before doing anything else.

## Day-of: one command

On the day, run only this. It checks whether the files are the known dataset (SAME DATA) or a new draw (NEW DATA), fits the frozen candidates and writes the upload files into `submissions/`. Everything below is the walk-through of what the model does.

In [ ]:
!cd .. && uv run python scripts/arena_run.py --practice --cv none  # --practice: data/raw still wins when it exists

In [ ]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from arena.config import ARCADE_ALGO_ARENA, TaskSpec
from arena.data import list_data_files, load_arcade, truncate
from arena.paths import SIGNALS, SUBMISSIONS, default_data_dir

DATA_DIR = default_data_dir()  # data/raw on the day; override: DATA_DIR = Path("...")
SPEC = ARCADE_ALGO_ARENA       # CHANGE if the rules differ: halfway bar, exit bar, Sharpe factor, columns
print(DATA_DIR)
list_data_files(DATA_DIR)

## 1 · Load + sanity checks (does the format match what we prepared for?)

In [ ]:
d = load_arcade(DATA_DIR)
ins, oos, y = d.ins, d.oos, d.y
print(f"in-sample     {ins.n} sessions x {ins.n_bars} bars, {len(ins.headlines)} headlines")
print(f"out-of-sample {oos.n} sessions x {oos.n_bars} bars, {len(oos.headlines)} headlines")
raw = ins.close[:, SPEC.exit_bar] / ins.close[:, SPEC.halfway_bar] - 1
print("returns = demeaned close(exit)/close(halfway)-1 ?", np.allclose(raw - raw.mean(), y))
print(f"return mean {y.mean():.2e}  sd {y.std():.4f}")

In [ ]:
s0 = 0
hl0 = ins.headlines[ins.headlines["sidx"] == s0]
for bar, text in zip(hl0["bar_ix"], hl0["headline"], strict=True):
    print(bar, text)
plt.plot(ins.close[s0]); plt.axvline(SPEC.halfway_bar, ls="--"); plt.axvline(SPEC.exit_bar, ls="--")
plt.title(f"session {ins.ids[s0]}"); plt.show()

## 2 · Headline templates + event study (how does a headline move the price?)

In [ ]:
from arena.events import event_study
from arena.models import ensure_keys

ins, oos = ensure_keys(ins), ensure_keys(oos)
print("distinct templates:", ins.headlines["key"].nunique())
es = event_study(ins, horizon=20)
path = es.drop(columns="n").mul(np.sign(es[10]), axis=0).mean() * 1e4
path.plot(marker="o", title="average |impact| after a headline, bp (ramp = drift, flat = done)")
plt.axvline(10, ls="--"); plt.show()
es.assign(bp_10=es[10] * 1e4)[["n", "bp_10"]].sort_values("bp_10")

## 3 · Baselines on the TEAM folds (numbers are comparable across the team)

In [ ]:
from arena.cv import signal_fit_predict
from arena.models import DriftModel, as_fit_predict
from arena.signals import team_folds
from arena.text import keyword_score
from arena.validation import compare, evaluate

folds = team_folds(ins.n)  # used by the bootstrap cell below
seen = truncate(ins, SPEC.halfway_bar)
kw = np.bincount(seen.headlines["sidx"], weights=keyword_score(seen.headlines["headline"]), minlength=ins.n)
kw_eval = evaluate("keyword", signal_fit_predict(kw), y)  # 5-fold x seeds 42/43/44
drift = DriftModel(SPEC, vol_power=1.0)
drift_eval = evaluate("drift", as_fit_predict(drift, ins, y), y)
print(kw_eval)
print(drift_eval)
c = compare(drift_eval, kw_eval, y)
print(f"drift vs keyword: Δ {c.delta:+.3f} (z {c.z:.1f}) keep={c.keep}")

## 4 · Industry filter (+0.3 Sharpe on the practice data)

Companies form groups that appear together; headlines about companies outside a session's main group barely move it.
Check the group sizes: on practice data it was 5 × 16. If they look unbalanced, try another `n_groups`.

In [ ]:
from arena.structure import cooccurrence_groups, keep_headlines, label_entities, mark_main_group

ins_e, oos_e = label_entities(ins, oos)
groups = cooccurrence_groups(ins_e, oos_e, n_groups=5)
print("group sizes:", groups.value_counts().sort_index().to_dict())
ins_m = keep_headlines(mark_main_group(ins_e, groups), "in_main")
oos_m = keep_headlines(mark_main_group(oos_e, groups), "in_main")
res = evaluate("drift + industry", as_fit_predict(drift, ins_m, y), y)
print(res)
c = compare(res, drift_eval, y)
print(f"industry vs drift: Δ {c.delta:+.3f} (z {c.z:.1f}) keep={c.keep}")

## 5 · Is the number real? (bootstrap + standard error)

In [ ]:
from arena.cv import out_of_fold
from arena.metrics import bootstrap_sharpe, sharpe_standard_error, summary

oof = out_of_fold(as_fit_predict(drift, ins_m, y), ins.n, folds)
print("bootstrap 5/50/95%:", np.round(bootstrap_sharpe(oof, y, SPEC), 2))
print("1 standard error  :", round(sharpe_standard_error(res.mean, ins.n, SPEC.annualization), 2),
      "-> differences smaller than this are noise")
pd.Series(summary(oof, y, SPEC)).round(4)

## 6 · Fit on all in-sample, predict out-of-sample, write the upload file (writing does NOT spend an upload)

In [ ]:
from arena.submission import log_submission, write_submission

fitted = drift.fit(ins_m, y)
pred = fitted.predict(oos_m)
path = write_submission(oos.ids, pred, SUBMISSIONS / "01_drift_industry.csv", SPEC)
print(path)
pd.read_csv(path).describe()

In [ ]:
# AFTER uploading: record it (one row per upload, even failed ones)
# log_submission(SUBMISSIONS / "ledger.csv", file=path.name, label="drift+industry",
#                cv_sharpe=res.mean, author="me", leaderboard=None, notes="")